In [1]:
import matplotlib
import matplotlib.pyplot as plt

from functools import reduce

import numpy as np
import pandas as pd
import geopandas as gpd
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from libpysal.weights import Queen, KNN
from libpysal import weights
from esda.moran import Moran
import spreg

from air_brain.asthma import build_dataset
from air_brain.tables import ols2df, spreg2df, p_col_style

In [2]:
df = build_dataset()
dep_vars = ["ED_visits_frac", "ED_hosp_frac", "UC_visits_frac", "Asthma_use_frac"]
indep_vars = ["PM25", "dpm", "lowincome", "poc"]

In [3]:
indep_to_use = {"ED_visits_frac": ["PM25", "dpm"],
                "ED_hosp_frac": ["PM25", "dpm"],
                "UC_visits_frac": ["PM25"],
                "Asthma_use_frac": ["PM25"]}
demos = ["lowincome", "poc"]

In [4]:
RADIUS_EARTH_MILES = 3959

In [5]:
def make_lag(df, col, weight_class, weight_kwargs=dict()):
    # spatial lag weights
    w = weight_class.from_dataframe(df, use_index=False, **weight_kwargs)
    # standardize by rows to use averages instead of sums
    w.transform = "R"
    # compute the spatial lag of column col
    w_col = weights.spatial_lag.lag_spatial(w, df[col])
    w_series = pd.Series(w_col, name="w_{}".format(col), index=df.index)
    return w_series

def test_weights(df, indep_vars, dep_vars, weight_class, weight_kwargs=dict()):
    # spatial lag weights
    w = weight_class.from_dataframe(df, use_index=False, **weight_kwargs)
    # standardize by rows to use averages instead of sums
    w.transform = "R"

    # compute the spatial lag of all independent variables
    # using row-standardized weights defined above
    wx = df[indep_vars].apply(lambda y: weights.spatial_lag.lag_spatial(w, y))
    # rename columns to w_*
    wx = wx.rename(columns=lambda c: "w_" + c)
    # glue onto other data
    dfx= df.join(wx)

    # linear regression, predicting asthma encounter fraction using each variable individually
    all_res = list()
    for indep in indep_vars:
        ols_res = dict()
        for dep in dep_vars:
            mod = sm.OLS(dfx[dep], dfx[[indep, "w_{}".format(indep), "intercept"]])
            res = mod.fit()
            ols_res[dep] = res
        all_res.append(ols2df(ols_res)[["coef", "std err", "p", "R2"]])
    res_df = pd.concat(all_res)
    
    return w, dfx, res_df

## Choice of spatial lag weights

The first decision when analyzing spatial autocorrelation is what the spatial weights should be. This decision encodes the assumptions about how difference regions influence each other.

Ideally, the choice of spatial weights wouldn't influence the conclusions. Unfortunately, that is not the case in this dataset, as I will show below. 

There are two reasonable choices for spatial weights for this analysis:
- Queen's contiguity: regions that are adjacent to each other on an edge or vertex are related.
- Gaussian kernel: a region is related to all other regions, with the strength of the relationship decreasing as a Gaussian function with distance. This choice requires a choice of the "width" of the Gaussian, i.e. how fast the weights decay with distance.

I'm not including block weights, since I don't have natural blocking units in this dataset. I'm also not including KNN, because the size of the spatial regions varies so dramatically, i.e. the area occupied by one "neighbor" is extremely variable. Additionally, there are many possible distance weighting kernels that could be used, but Gaussian is the natural choice for dispersion of a contaminant in air.

### Queen's contiguity
What are the conclusions if we use Queen's contiguity for all weights?

First, check if the variables are spatially autocorrelated when using this weight.

In [6]:
# spatial lag weights from Queen contiguity
w_queen = weights.Queen.from_dataframe(df, use_index=False)
# standardize by rows to use averages instead of sums
w_queen.transform = "R"
# compute Moran's I
res_dict = {}
for col in ["PM25", "dpm", "lowincome", "poc"] + dep_vars:
    moran = Moran(df[col], w_queen)
    res_dict[col] = {"I": moran.I, "p": moran.p_sim}
res = pd.DataFrame(res_dict).transpose()
res

,I,p
PM25,0.957093,0.001
dpm,0.778897,0.001
lowincome,0.553072,0.001
poc,0.652880,0.001
ED_visits_frac,0.412046,0.001
ED_hosp_frac,0.308218,0.001
UC_visits_frac,0.139003,0.001
Asthma_use_frac,0.239554,0.001


All variables are significantly spatially autocorrelated. Note that Moran's I ranges between -1 (dissimilar values cluster) to 1 (similar values cluster). Air quality measures have extremely high values for Moran's I, particularly PM 2.5, as expected from mapping the variables.

What happens when we include spatial autocorrelation in the regression models? For simplicity, I will consider each variable individually. Recall that each independent variable, on its own, was a significant predictor of each outcome variable, with the exception of diesel PM for urgent care and overall asthma encounters.

In [7]:
# compute the spatial lag of all independent variables (air quality + demographics)
# using row-standardized weights defined above
wx_queen = df[indep_vars].apply(lambda y: weights.spatial_lag.lag_spatial(w_queen, y))
# rename columns to w_*
wx_queen = wx_queen.rename(columns=lambda c: "w_" + c)
# glue onto other data
dfx_queen = df.join(wx_queen)

In [8]:
# linear regression, predicting asthma encounter fraction using each variable individually
all_res = list()
for indep in indep_vars:
    ols_res = dict()
    for dep in dep_vars:
        mod = sm.OLS(dfx_queen[dep], dfx_queen[[indep, "w_{}".format(indep), "intercept"]])
        res = mod.fit()
        ols_res[dep] = res
    all_res.append(ols2df(ols_res)[["coef", "std err", "p", "R2"]])
res_queen = pd.concat(all_res)
# there are a lot of rows here, so subset to significant results
res_queen.loc[res_queen.p < 0.05].style.pipe(p_col_style)

None of the original air quality variables are significant predictors anymore, and neither are their spatial lags, except for the spatial lag in diesel PM predicting ED outcomes. This result suggets that the previously seen significant relationship between air quality and asthma outcomes was due to the spatial autocorrelation in air quality measures, and not a real result.

Lowincome and its spatial lag are significant predictors of all outcomes except general asthma use.

PoC is a significant predictor of all outcomes except urgent care, and its spatial lag is also a significant predictor of ED visits.

### Gaussian kernel: fast decay
Distance weighting with a Gaussian kernel requires the choice of how fast the spatial lag decays with distance. I'm going to try two different weights, for a "fast" and "slow" decay.

The fast decay has a bandwidth of 5 miles. For reference, Allegheny County is 745 square miles, sqrt(745) = 27.3.

In [9]:
# spatial lag weights from Gaussian kernel, with fixed fast decay / narrow bandwidth
w_fast = weights.Kernel.from_dataframe(df, use_index=False, fixed=True, function="gaussian", bandwidth=5,
                                      distance_metric="arc", radius=RADIUS_EARTH_MILES, # because geometry is in latitude/longitude
                                     )
# standardize by rows to use averages instead of sums
w_fast.transform = "R"
# compute Moran's I
res_dict = {}
for col in ["PM25", "dpm", "lowincome", "poc"] + dep_vars:
    moran = Moran(df[col], w_fast)
    res_dict[col] = {"I": moran.I, "p": moran.p_sim}
res = pd.DataFrame(res_dict).transpose()
res

,I,p
PM25,0.793258,0.001
dpm,0.473050,0.001
lowincome,0.290480,0.001
poc,0.264865,0.001
ED_visits_frac,0.169004,0.001
ED_hosp_frac,0.131561,0.001
UC_visits_frac,0.089154,0.001
Asthma_use_frac,0.071853,0.002


Again, all the variables are spatially autocorrelated. But the Moran's I values are lower than seen for Queen's contiguity.

In [10]:
# compute the spatial lag of all independent variables (air quality + demographics)
# using row-standardized weights defined above
wx_fast = df[["PM25", "dpm", "lowincome", "poc"]].apply(lambda y: weights.spatial_lag.lag_spatial(w_fast, y))
# rename columns to w_*
wx_fast = wx_fast.rename(columns=lambda c: "w_" + c)
# glue onto other data
dfx_fast= df.join(wx_fast)

In [11]:
# linear regression, predicting asthma encounter fraction using each variable individually
all_res = list()
for indep in indep_vars:
    ols_res = dict()
    for dep in dep_vars:
        mod = sm.OLS(dfx_fast[dep], dfx_fast[[indep, "w_{}".format(indep), "intercept"]])
        res = mod.fit()
        ols_res[dep] = res
    all_res.append(ols2df(ols_res)[["coef", "std err", "p", "R2"]])
res_fast = pd.concat(all_res)
# there are a lot of rows here, so subset to significant results
res_fast.loc[res_fast.p < 0.05].style.pipe(p_col_style)

With these weights, PM 2.5 is still a significant predictor of the ED outcomes. The other results are similar to using Queen's contiguity.

### Gaussian kernel: slow decay
Bandwidth of 25 miles.

In [20]:
# spatial lag weights from Gaussian kernel, with fixed slow decay/wide bandwidth
w_slow = weights.Kernel.from_dataframe(df, use_index=False, fixed=True, function="gaussian", bandwidth=25,
                                      distance_metric="arc", radius=RADIUS_EARTH_MILES, # because geometry is in latitude/longitude
                                     )
# standardize by rows to use averages instead of sums
w_slow.transform = "R"
# compute Moran's I
res_dict = {}
for col in ["PM25", "dpm", "lowincome", "poc"] + dep_vars:
    moran = Moran(df[col], w_slow)
    res_dict[col] = {"I": moran.I, "p": moran.p_sim}
res = pd.DataFrame(res_dict).transpose()
res

,I,p
PM25,0.018645,0.001
dpm,-0.003215,0.001
lowincome,0.002825,0.001
poc,0.000186,0.476
ED_visits_frac,0.001459,0.015
ED_hosp_frac,0.001037,0.038
UC_visits_frac,0.003088,0.001
Asthma_use_frac,0.000405,0.285


Now two variables are no longer spatially autocorrelated -- PoC and the general asthma outcome. Moreover, diesel PM is now negatively spatially autocorrelated. The values for Moran's I are also much closer to 0, showing that this spatial weighting shows a lesser degree of spatial autocorrelation even for significant variables.

In [13]:
# compute the spatial lag of all independent variables (air quality + demographics)
# using row-standardized weights defined above
wx_slow = df[["PM25", "dpm", "lowincome", "poc"]].apply(lambda y: weights.spatial_lag.lag_spatial(w_slow, y))
# rename columns to w_*
wx_slow = wx_slow.rename(columns=lambda c: "w_" + c)
# glue onto other data
dfx_slow = df.join(wx_slow)

In [14]:
# linear regression, predicting asthma encounter fraction using each variable individually
all_res = list()
for indep in indep_vars:
    ols_res = dict()
    for dep in dep_vars:
        mod = sm.OLS(dfx_slow[dep], dfx_slow[[indep, "w_{}".format(indep), "intercept"]])
        res = mod.fit()
        ols_res[dep] = res
    all_res.append(ols2df(ols_res)[["coef", "std err", "p", "R2"]])
res_slow = pd.concat(all_res)
# there are a lot of rows here, so subset to significant results
res_slow.loc[res_slow.p < 0.05].style.pipe(p_col_style)

For these weights, almost all previously significant variables remain significant.

### How to choose
Based on the underlying processes, I would choose
- Queen's contiguity for lowincome and poc
    - This choice assumes that the fraction of low income and PoC residents in a census tract is influenced by the respective fraction of residents in directly neighboring census tracts. This assumption is extremely commonly used in the literature for demographic effects [CITATION NEEDED].
- Gaussian kernel distance for PM 2.5
    - Here I am assuming that the concentration of PM 2.5 in one location is related to the concentration in another by [diffusion](https://en.wikipedia.org/wiki/Diffusion#Basic_models_of_diffusion). This assumption is not entirely correct, since many factors such as overall wind direction and topography will affect [how air pollutants disperse](https://en.wikipedia.org/wiki/Atmospheric_dispersion_modeling).
    - I can estimate the appropriate width of the Gaussian kernel by the observation of how PM 2.5 is spatially distributed in the data.
- KNN or Queen's contiguity for diesel PM
    - The dispersion of diesel PM isn't the same as that of PM 2.5, as is evident from observing the maps of their distributions. While PM 2.5 largely comes from industrial stacks, diesel PM largely comes from vehicle traffic. This means that diesel PM doesn't spread as far, and clusters in census tracts near major highways. Either KNN or Queen's contiguity would be appropriate choices to model short distance dispersion.

However, because my results are so different depending on what spatial weighting I choose, I want to be methodical in choosing a spatial weight. As I did to choose a method of combining variables to reduce collinearity, I will use the choice that gives the largest coefficient of determination for each individual regression, which may differ across independent/dependent variable pairings.

In [15]:
to_test = {"queen": [weights.Queen, dict()],
           #"KNN_10": [weights.KNN, {"k": 10}],
           #"KNN_20": [weights.KNN, {"k": 20}],
           #"KNN_30": [weights.KNN, {"k": 30}],
           #"KNN_40": [weights.KNN, {"k": 40}],
           #"KNN_100": [weights.KNN, {"k": 100}],
           "Gaussian_5": [weights.Kernel, {"function": "gaussian", "bandwidth": 5, "distance_metric": "arc", "radius": RADIUS_EARTH_MILES}],
           "Gaussian_10": [weights.Kernel, {"function": "gaussian", "bandwidth": 10, "distance_metric": "arc", "radius": RADIUS_EARTH_MILES}],
           "Gaussian_15": [weights.Kernel, {"function": "gaussian", "bandwidth": 15, "distance_metric": "arc", "radius": RADIUS_EARTH_MILES}],
           "Gaussian_20": [weights.Kernel, {"function": "gaussian", "bandwidth": 20, "distance_metric": "arc", "radius": RADIUS_EARTH_MILES}],
           "Gaussian_25": [weights.Kernel, {"function": "gaussian", "bandwidth": 25, "distance_metric": "arc", "radius": RADIUS_EARTH_MILES}],
           "Gaussian_30": [weights.Kernel, {"function": "gaussian", "bandwidth": 30, "distance_metric": "arc", "radius": RADIUS_EARTH_MILES}],
           "Gaussian_35": [weights.Kernel, {"function": "gaussian", "bandwidth": 30, "distance_metric": "arc", "radius": RADIUS_EARTH_MILES}],
          }

In [16]:
dfs = []
for name, params in to_test.items():
    _, _, res_df = test_weights(df, indep_vars, dep_vars, params[0], weight_kwargs=params[1])
    dfs.append(res_df[["R2"]].rename(columns={"R2": name}))
all_res = reduce(lambda x,y: pd.merge(x, y, left_index=True, right_index=True), dfs)
all_res["choice"] = all_res.idxmax(axis=1)
all_res.loc[~all_res.index.get_level_values(1).str.contains("w_")]

,,queen,Gaussian_5,Gaussian_10,Gaussian_15,Gaussian_20,Gaussian_25,Gaussian_30,Gaussian_35,choice
dep_var,indep_var,,,,,,,,,
ED_visits_frac,PM25,0.130192,0.131034,0.134654,0.138222,0.133504,0.130964,0.132251,0.132251,Gaussian_15
ED_hosp_frac,PM25,0.098920,0.100131,0.104704,0.106919,0.102781,0.100019,0.101060,0.101060,Gaussian_15
UC_visits_frac,PM25,0.037390,0.040390,0.038681,0.043326,0.051013,0.052702,0.052940,0.052940,Gaussian_30
Asthma_use_frac,PM25,0.070043,0.071372,0.071549,0.071001,0.070794,0.072108,0.072259,0.072259,Gaussian_30
ED_visits_frac,dpm,0.100996,0.119505,0.097213,0.092439,0.098196,0.094543,0.091829,0.091829,Gaussian_5
ED_hosp_frac,dpm,0.065898,0.092845,0.061537,0.054178,0.062688,0.055126,0.052469,0.052469,Gaussian_5
UC_visits_frac,dpm,0.000919,0.001385,0.000228,0.001379,0.000327,0.000221,0.003857,0.003857,Gaussian_30
Asthma_use_frac,dpm,0.001287,0.010294,0.011087,0.005630,0.007313,0.018888,0.004758,0.004758,Gaussian_25
ED_visits_frac,lowincome,0.412945,0.380064,0.374331,0.368555,0.367911,0.368164,0.369918,0.369918,queen


As expected, PM 2.5 and its lag predict all outcomes best when using a Gaussian kernel spatial lag. The ED outcomes are fit better with a smaller bandwidth than the urgent care and general asthma use outcomes, although the differences in R2 are not large. (TODO are these differences significant??? for any of these). 

Similarly, diesel PM and its lag are best predicted with a Gaussian kernel spatial lag, with a smaller bandwidth for ED. The bandwidth is also smaller than the bandwidth for PM 2.5, matching my understanding of the physical processes.

Lowincome is best used with queen contiguity for all outcomes except urgent care (more on this below). PoC wasn't best paired with Queen, but are those differences actually significant??

A Gaussian kernel with bandwith of 30 miles was the best spatial lag for the urgent care outcome for all variables. WHY???

In [17]:
lag_dict = {"PM25": "Gaussian_15",
            "dpm": "Gaussian_5",
            "lowincome": "queen",
            "poc": "queen"}

In [18]:
w_df = df.copy()
for col, lag in lag_dict.items():
    new_col = make_lag(df, col, to_test[lag][0], to_test[lag][1])
    w_df = pd.concat([w_df, new_col], axis=1)

In [19]:
# linear regression, predicting asthma encounter fraction using each variable individually
all_res = list()
for indep in indep_vars:
    ols_res = dict()
    for dep in dep_vars:
        mod = sm.OLS(w_df[dep], w_df[[indep, "w_{}".format(indep), "intercept"]])
        res = mod.fit()
        ols_res[dep] = res
    all_res.append(ols2df(ols_res)[["coef", "std err", "p", "R2"]])
res_df = pd.concat(all_res)
# there are a lot of rows here, so subset to significant results
res_df.loc[res_df.p < 0.05].style.pipe(p_col_style)